# Chapter 01 - Functions

Functions are the primary way to organize and reuse code in Python. This notebook covers
how to define functions, work with different parameter types, understand variable scope,
and leverage functional programming tools like `map()`, `filter()`, and `lambda`.

**Topics covered:**
- `def` syntax and calling functions
- Parameters and arguments (`*args`, `**kwargs`)
- Return values
- Scope and the LEGB rule
- Lambda functions
- `map()`, `filter()`, `zip()`, `sorted()` with `key`
- Nested functions
- Docstrings
- Type hints basics

## Defining Functions with `def`

A function is defined with the `def` keyword, followed by the function name,
parameters in parentheses, and a colon. The body is indented.

In [ ]:
def greet(name):
    """Return a greeting string for the given name."""
    return f'Hello, {name}!'

print(greet('Alice'))
print(greet('Bob'))

In [ ]:
# A function with no explicit return statement returns None
def say_hello(name):
    print(f'Hello, {name}!')

result = say_hello('Carol')
print(f'Return value: {result}')

## Parameters and Arguments

Python supports several kinds of parameters:

| Kind | Syntax | Description |
|------|--------|-------------|
| Positional | `def f(a, b)` | Matched by position |
| Default | `def f(a, b=10)` | Falls back to default if not provided |
| Keyword | `f(b=5, a=3)` | Matched by name at call site |
| `*args` | `def f(*args)` | Collects extra positional args into a tuple |
| `**kwargs` | `def f(**kwargs)` | Collects extra keyword args into a dict |

In [ ]:
# Positional and default parameters
def power(base, exponent=2):
    """Raise base to the given exponent (default: squared)."""
    return base ** exponent

print(power(5))        # 25  (exponent defaults to 2)
print(power(2, 10))    # 1024
print(power(3, exponent=3))  # 27  (keyword argument)

In [ ]:
# *args collects extra positional arguments into a tuple
def total(*args):
    """Return the sum of all arguments."""
    print(f'Received: {args} (type: {type(args).__name__})')
    return sum(args)

print(total(1, 2, 3))
print(total(10, 20, 30, 40, 50))

In [ ]:
# **kwargs collects extra keyword arguments into a dictionary
def build_profile(name, **kwargs):
    """Build a user profile dictionary."""
    profile = {'name': name}
    profile.update(kwargs)
    return profile

user = build_profile('Alice', age=28, field='data science', level='senior')
print(user)

In [ ]:
# Combining all parameter types
def flexible(required, default=10, *args, **kwargs):
    print(f'required: {required}')
    print(f'default:  {default}')
    print(f'args:     {args}')
    print(f'kwargs:   {kwargs}')

flexible('hello', 20, 'a', 'b', 'c', x=1, y=2)

> **Pro tip:** Avoid using mutable default arguments (like lists or dicts).
> The default is shared across all calls, which leads to unexpected behavior.
> Use `None` instead and create the mutable object inside the function.

In [ ]:
# BAD: mutable default argument
def append_to_bad(item, target=[]):
    target.append(item)
    return target

print(append_to_bad(1))  # [1]
print(append_to_bad(2))  # [1, 2]  <-- unexpected!

# GOOD: use None as sentinel
def append_to_good(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target

print(append_to_good(1))  # [1]
print(append_to_good(2))  # [2]  <-- correct

## Return Values

Functions can return any object. You can return multiple values using a tuple
(with or without parentheses), which is very common in data science code.

In [ ]:
def describe(numbers):
    """Return basic statistics for a list of numbers."""
    n = len(numbers)
    total = sum(numbers)
    mean = total / n
    sorted_nums = sorted(numbers)
    minimum = sorted_nums[0]
    maximum = sorted_nums[-1]
    return mean, minimum, maximum

data = [23, 45, 12, 67, 34, 89, 5]
avg, lo, hi = describe(data)
print(f'Mean: {avg:.1f}, Min: {lo}, Max: {hi}')

## Scope and the LEGB Rule

When Python encounters a variable name, it searches these scopes in order:

1. **L**ocal — inside the current function
2. **E**nclosing — inside any enclosing (outer) function
3. **G**lobal — at the module level
4. **B**uilt-in — Python's built-in names (`len`, `print`, `range`, etc.)

In [ ]:
x = 'global'

def outer():
    x = 'enclosing'

    def inner():
        x = 'local'
        print(f'inner sees: {x}')

    inner()
    print(f'outer sees: {x}')

outer()
print(f'module sees: {x}')

In [ ]:
# The 'global' keyword lets a function modify a module-level variable
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(f'counter = {counter}')  # 2

# The 'nonlocal' keyword lets an inner function modify an enclosing variable
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

my_counter = make_counter()
print(my_counter())  # 1
print(my_counter())  # 2
print(my_counter())  # 3

> **Pro tip:** Avoid `global` in production code. If you need shared mutable state,
> consider using a class, a closure (like `make_counter` above), or passing values
> explicitly through function arguments.

## Lambda Functions

A `lambda` is a small anonymous function defined in a single expression.
They are most useful as short callbacks passed to higher-order functions.

In [ ]:
# Lambda syntax: lambda parameters: expression
square = lambda x: x ** 2
print(square(5))  # 25

# Multiple parameters
add = lambda a, b: a + b
print(add(3, 7))  # 10

# Lambdas shine as inline arguments
pairs = [(1, 'banana'), (3, 'apple'), (2, 'cherry')]
pairs_sorted = sorted(pairs, key=lambda p: p[1])
print('Sorted by name:', pairs_sorted)

## `map()` and `filter()`

- **`map(func, iterable)`** — apply a function to every element
- **`filter(func, iterable)`** — keep only elements where the function returns truthy

Both return lazy iterators (wrap with `list()` to see results).

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map: square every number
squared = list(map(lambda x: x ** 2, numbers))
print('Squared:', squared)

# filter: keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print('Evens:  ', evens)

# Combine map and filter
even_squares = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, numbers)))
print('Even squares:', even_squares)

# The comprehension equivalent is often clearer
even_squares_v2 = [x ** 2 for x in numbers if x % 2 == 0]
print('Same result: ', even_squares_v2)

## `sorted()` with `key`

The `key` parameter accepts a function that extracts a comparison key from each element.
This is extremely useful for custom sorting.

In [ ]:
# Sort strings by length
words = ['banana', 'pie', 'watermelon', 'kiwi', 'fig']
by_length = sorted(words, key=len)
print('By length:', by_length)

# Sort by absolute value
values = [-5, 3, -1, 4, -2]
by_abs = sorted(values, key=abs)
print('By abs:   ', by_abs)

# Sort dicts by a specific field
students = [
    {'name': 'Alice', 'gpa': 3.8},
    {'name': 'Bob', 'gpa': 3.5},
    {'name': 'Carol', 'gpa': 3.9},
]
by_gpa = sorted(students, key=lambda s: s['gpa'], reverse=True)
for s in by_gpa:
    print(f"  {s['name']}: {s['gpa']}")

## `zip()` Revisited

`zip()` is a built-in that pairs elements from multiple iterables.
Combined with functions, it becomes a powerful data-processing tool.

In [ ]:
names = ['Alice', 'Bob', 'Carol']
scores = [92, 85, 97]

# Unzipping with zip(*...)
pairs = list(zip(names, scores))
print('Zipped:', pairs)

unzipped_names, unzipped_scores = zip(*pairs)
print('Names: ', unzipped_names)
print('Scores:', unzipped_scores)

# Element-wise operations
a = [1, 2, 3]
b = [10, 20, 30]
sums = [x + y for x, y in zip(a, b)]
print('Element-wise sums:', sums)

## Nested Functions

Functions can be defined inside other functions. This is useful for:
- Helper logic that only makes sense in one context
- Closures that capture state from the enclosing scope
- Factory functions that return specialized functions

In [ ]:
# Factory function: create a multiplier
def make_multiplier(factor):
    """Return a function that multiplies its input by factor."""
    def multiplier(x):
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

# The inner function "remembers" the factor from its enclosing scope
print(double(100))  # 200

## Docstrings

A docstring is a string literal that appears as the first statement in a function,
class, or module. It is stored in the `__doc__` attribute and used by `help()`.

Follow the [Google-style](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings)
or [NumPy-style](https://numpydoc.readthedocs.io/en/latest/format.html) conventions for consistency.

In [ ]:
def normalize(values, method='min-max'):
    """Normalize a list of numbers.

    Args:
        values: A list of numeric values.
        method: Normalization method, either 'min-max' or 'z-score'.

    Returns:
        A new list with normalized values.

    Raises:
        ValueError: If an unknown method is provided.
    """
    if method == 'min-max':
        lo, hi = min(values), max(values)
        span = hi - lo
        return [(v - lo) / span for v in values] if span else [0.0] * len(values)
    elif method == 'z-score':
        n = len(values)
        mean = sum(values) / n
        std = (sum((v - mean) ** 2 for v in values) / n) ** 0.5
        return [(v - mean) / std for v in values] if std else [0.0] * len(values)
    else:
        raise ValueError(f'Unknown method: {method!r}')

data = [10, 20, 30, 40, 50]
print('min-max:', [round(v, 2) for v in normalize(data)])
print('z-score:', [round(v, 2) for v in normalize(data, 'z-score')])

# Access the docstring
help(normalize)

## Type Hints Basics

Type hints (PEP 484) let you annotate function signatures with expected types.
They do **not** enforce types at runtime — they are used by tools like `mypy`,
IDEs, and linters to catch bugs early.

In [ ]:
def mean(values: list[float]) -> float:
    """Calculate the arithmetic mean of a list of numbers."""
    return sum(values) / len(values)

def format_result(name: str, value: float, decimals: int = 2) -> str:
    """Format a named result with the given number of decimal places."""
    return f'{name}: {value:.{decimals}f}'

data = [23.5, 45.1, 12.8, 67.3]
avg = mean(data)
print(format_result('Average', avg))
print(format_result('Average', avg, decimals=4))

In [ ]:
# Common type hint patterns
from typing import Optional

def find_max(values: list[float], threshold: Optional[float] = None) -> Optional[float]:
    """Return the max value, or None if no value exceeds the threshold."""
    if threshold is not None:
        values = [v for v in values if v > threshold]
    return max(values) if values else None

print(find_max([1, 5, 3, 8, 2]))           # 8
print(find_max([1, 5, 3, 8, 2], threshold=6))  # 8
print(find_max([1, 5, 3], threshold=10))    # None

> **Note:** From Python 3.10+, you can use `float | None` instead of `Optional[float]`,
> and `list[float]` has been valid since Python 3.9 without needing `from typing import List`.

---

**Summary:** This notebook covered how to define and call functions, work with all parameter
types (`*args`, `**kwargs`, defaults), understand scope (LEGB), use lambda functions and
functional programming tools (`map`, `filter`, `sorted`), create nested functions and closures,
write proper docstrings, and annotate code with type hints.

**Next chapter:** NumPy — numerical computing with arrays.